# 01: Data Cleaning & Transformation
***
**Author:** Zain Zabidi
**Date:** February 2026

**Objective:** To clean raw Census CSVs, filter for Western Australian data, and apply midpoint mappings to create a consolidated numerical dataset for analysis.

## 1. Environment Setup & Data Loading
First we import the raw Census CSVs and the midpoint dictionaries.

**Datasets to be processed:**
* **G01 & G02:** Baseline population characteristics and official ABS medians for validation.
* **G33, G38 , G40:** Categorical data requiring midpoint mapping (Income, Mortgage, Rent).
* **G46, G54:** Count-based data (Labour Force, Industry).

### 1.1 Import Libraries and Define Paths
Using **Pandas** for transformation and **OS/JSON** for file management and dictionary retrieval.

In [16]:
import pandas as pd
import os
import json

# Set directory paths for raw data, dictionaries, and processed output
RAW_DATA_PATH = '../data/raw/'
DICT_PATH = '../data/dictionary/'
PROCESSED_DATA_PATH = '../data/processed/'

# Ensure processed directory exists
os.makedirs(PROCESSED_DATA_PATH, exist_ok=True)

# List all required ABS Census tables for the project
G_FILES = [
    'G01', 'G02', 'G33', 'G38', 'G40', 'G46A', 'G46B', 'G54A', 'G54B'
]

# Mapping dicts created in notebook 00 to respective data types
DICT_FILES = {
    'income': 'income_midpoints.json',
    'mortgage': 'mortgage_midpoints.json',
    'rent': 'rent_midpoints.json'
}

### 1.2 Data Ingestion
We load the CSV files into a dictionary of DataFrames.These datasets were acquired already pre-filtered for Western Australia.

In [3]:
# Dict to store the filtered DataFrames
dfs = {}

for g_file in G_FILES:
    # Construct file path
    file_name = f'2021Census_{g_file}_WA_SAL.csv'
    full_path = os.path.join(RAW_DATA_PATH, file_name)

    # Load dataset
    dfs[g_file] = pd.read_csv(full_path)

# Verify number of rows in samples
for name, df in dfs.items():
    print(f"{name}: {df.shape[0]} records")

G01: 1701 records
G02: 1701 records
G33: 1701 records
G38: 1701 records
G40: 1701 records
G41: 1701 records
G46A: 1701 records
G46B: 1701 records
G54A: 1701 records
G54B: 1701 records


### 1.3 Load Mapping Dictionaries
We load the midpoint dictionaries previously exported to JSON. These will be used to transform categorical range data into continuous numerical values.

In [6]:
# Dict to store loaded midpoint mappings
midpoint_dicts = {}

for key, filename in DICT_FILES.items():
    dict_full_path = os.path.join(DICT_PATH, filename)

    with open(dict_full_path, 'r') as f:
        midpoint_dicts[key] = json.load(f)

# Check data loaded correctly
for key, mapping in midpoint_dicts.items():
    sample = dict(list(mapping.items())[:3])
    print(f"{key.capitalize()} mapping sample: {sample}")

Income mapping sample: {'Negative_Nil_income_Tot': 0, 'HI_1_149_Tot': 75.0, 'HI_150_299_Tot': 224.5}
Mortgage mapping sample: {'M_0_299_Tot': 149.5, 'M_300_449_Tot': 374.5, 'M_450_599_Tot': 524.5}
Rent mapping sample: {'R_1_74_Tot': 37.5, 'R_75_99_Tot': 87.0, 'R_100_149_Tot': 124.5}


## Section 2: Data Transformation
In this section, we apply the midpoint mappings to the Census tables. We will process each major variable (Rent, Mortgage, Income) individually to ensure accuracy before merging the features into a single dataset.

### 2.1 Process Weekly Rent (G40)
We apply the rent midpoint dictionary to the G40 table. This allows us to estimate the total weekly rent expenditure per suburb by multiplying household counts by bracket midpoints.

**Formula:**
$$Total\_Rent\_Weekly = \sum (Household\_Count_{range} \times Midpoint_{range})$$

*Where:*
* ***Household_Count:** Number of households reported in a specific rent range (e.g., $150-$199).*
* ***Midpoint:** The numerical representative value for that range (e.g., $174.5).*
* ***Total:** The sum of all rent paid by all reporting households in the suburb combined.*

In [9]:
# Identify columns in G40 that match to our rent midpoint keys
rent_cols = list(midpoint_dicts['rent'].keys())

# Create copy of G40 dataframe to alter
rent_df = dfs['G40'].copy()

# Calculate estimated total rent for each bracket
# Multiply household count by midpoint value
for col in rent_cols:
    midpoint = midpoint_dicts['rent'][col]
    rent_df[f'{col}_val'] = rent_df[col] * midpoint

# Sum calculated values for single 'total_rent_weekly' column
val_cols = [f'{col}_val' for col in rent_cols]
rent_df['total_rent_weekly'] = rent_df[val_cols].sum(axis=1)

# Display check for first few results
display(rent_df[['SAL_CODE_2021', 'total_rent_weekly']].head())

,SAL_CODE_2021,total_rent_weekly
0,SAL50001,0.0
1,SAL50002,38475.5
2,SAL50003,998.0
3,SAL50004,0.0
4,SAL50005,0.0


**Observation:**
The `total_rent_weekly` value represents the estimated aggregate rent for the entire suburb.
* For example, **SAL50002** shows ~$38,475, which is the sum of all rent paid by all households that fell within our defined price ranges.
* Suburbs with **0.0** (like SAL50001) indicate areas with no reported rental households in those specific Census categories (likely industrial areas, parks, or zero-population zones).

This aggregate figure serves as our **numerator** for calculating the mean weekly rent in the next step.

### 2.2 Calculate Renting Households
To calculate a meaningful average, we need to know how many households contributed to the total rent expenditure. We sum the counts from the same price ranges used in the previous step.

In [10]:
# Sum counts of households in relevant rent price ranges
rent_df['renting_households_count'] = rent_df[rent_cols].sum(axis=1)

# Calculate Average Weekly Rent per Suburb
# Avoid dividing by zero for no-renter suburbs with a small check
rent_df['avg_weekly_rent'] = (
    rent_df['total_rent_weekly'] / rent_df['renting_households_count']
).fillna(0)

# Display check
display(rent_df[['SAL_CODE_2021', 'total_rent_weekly', 'renting_households_count', 'avg_weekly_rent']].head())

,SAL_CODE_2021,total_rent_weekly,renting_households_count,avg_weekly_rent
0,SAL50001,0.0,0,0.000000
1,SAL50002,38475.5,104,369.956731
2,SAL50003,998.0,4,249.500000
3,SAL50004,0.0,0,0.000000
4,SAL50005,0.0,0,0.000000


### 2.3 Process Monthly Mortgage (G38)
Table G38 breaks down mortgage repayments by dwelling structure (House, Townhouse, etc.). We first aggregate these sub-categories into total columns for each price range, then apply our midpoint mappings to calculate the average monthly mortgage per suburb.

In [18]:
# Create copy of the G38 dataframe
mortgage_df = dfs['G38'].copy()

# Identify mortgage brackets from dict
mort_cols = list(midpoint_dicts['mortgage'].keys())

# Sum dwelling-specific columns to create aggregate Total columns
for target_col in mort_cols:
    base_range = target_col.replace('_Tot', '')
    sub_cols = [c for c in mortgage_df if c.startswith(base_range)]
    mortgage_df[target_col] = mortgage_df[sub_cols].sum(axis=1)

# Apply midpoints to calculate estimated expenditure for each price range
for col in mort_cols:
    midpoint = midpoint_dicts['mortgage'][col]
    mortgage_df[f'{col}_val'] = mortgage_df[col] * midpoint

# Calculate total monthly expenditure and mortgage-paying households count
val_cols_m = [f'{col}_val' for col in mort_cols]
mortgage_df['total_mortgage_monthly'] = mortgage_df[val_cols_m].sum(axis=1)
mortgage_df['mortgage_households_count'] = mortgage_df[mort_cols].sum(axis=1)

# Calculate average monthly mortgage per suburb
mortgage_df['avg_monthly_mortgage'] = (
    mortgage_df['total_mortgage_monthly'] / mortgage_df['mortgage_households_count']
).fillna(0)

# Display check
display(mortgage_df[['SAL_CODE_2021', 'total_mortgage_monthly', 'mortgage_households_count', 'avg_monthly_mortgage']].head())

,SAL_CODE_2021,total_mortgage_monthly,mortgage_households_count,avg_monthly_mortgage
0,SAL50001,54591.0,18,3032.833333
1,SAL50002,521859.5,288,1812.012153
2,SAL50003,32992.0,16,2062.000000
3,SAL50004,0.0,0,0.000000
4,SAL50005,0.0,0,0.000000


### 2.4 Process Weekly Household Income (G33)
We apply the income midpoint dictionary to the G33 tbale. Unlike the mortgage data, this table already provides pre-aggregated total columns for each income price range, allowing for a direct mapping to numerical values.

In [21]:
# Create copy of G33 dataframe
inc_df = dfs['G33'].copy()

# Identify income price range from dictionary
inc_cols = list(midpoint_dicts['income'].keys())

# Apply midpoints to calculate estimated income for each price range
for col in inc_cols:
    midpoint = midpoint_dicts['income'][col]
    inc_df[f'{col}_val'] = inc_df[col] * midpoint

# Calculate total weekly income & income-reporting households count
val_cols_i = [f'{col}_val' for col in inc_cols]
inc_df['total_income_weekly'] = inc_df[val_cols_i].sum(axis=1)
inc_df['income_households_count'] = inc_df[inc_cols].sum(axis=1)

# Calculate average weekly household income per suburb
inc_df['avg_weekly_income'] = (
    inc_df['total_income_weekly'] / inc_df['income_households_count']
).fillna(0)

# Display check
display(inc_df[['SAL_CODE_2021', 'total_income_weekly', 'income_households_count', 'avg_weekly_income']].head())

,SAL_CODE_2021,total_income_weekly,income_households_count,avg_weekly_income
0,SAL50001,41245.5,14,2946.107143
1,SAL50002,834671.0,456,1830.418860
2,SAL50003,38287.5,29,1320.258621
3,SAL50004,19200.0,4,4800.000000
4,SAL50005,1348.5,3,449.500000


### 2.5 Process Industry of Employment (G54A & G54B)
To address the research question regarding employment concentration, we extract industry totals from the G54 tables. Since male data is in G54A and female data is in G54B, we merge these tables and aggregate gendered counts to calculate total industry employment and the total workforce per suburb.

In [27]:
# Create copies of G54 tables
ind_a_df = dfs['G54A'].copy()
ind_b_df = dfs['G54B'].copy()

# Define specific industry columns for extraction
industry_targets = {
    'Mining': {'M': 'M_Mining_Tot', 'F': 'F_Mining_Tot'},
    'Education': {'M': 'M_Educ_trng_Tot', 'F': 'F_Educ_trng_Tot'},
    'Healthcare': {'M': 'M_HlthCare_SocAs_Tot', 'F': 'F_HlthCare_SocAs_Tot'},
    'Professional_Tech': {'M': 'M_Pro_scien_tec_Tot', 'F': 'F_Pro_scien_tec_Tot'}
}

# Merge gender-split tables
ind_df = pd.merge(ind_a_df, ind_b_df, on='SAL_CODE_2021')

# Calculate total persons per target industry
for industry, keys in industry_targets.items():
    ind_df[f'total_{industry.lower()}'] = ind_df[keys['M']] + ind_df[keys['F']]

# Aggregate all female industry totals to calculate total female workforce
f_tot_cols = [col for col in ind_b_df.columns if col.startswith('F_') and col.endswith('_Tot')]
ind_df['F_Total_Calculated'] = ind_df[f_tot_cols].sum(axis=1)

# Calculate total workforce (Male Total + Calculated Female Total)
ind_df['total_workforce'] = ind_df['M_Tot_Tot'] + ind_df['F_Total_Calculated']

# Validate output
display(ind_df[['SAL_CODE_2021', 'total_mining', 'total_education', 'total_workforce']].head())

,SAL_CODE_2021,total_mining,total_education,total_workforce
0,SAL50001,5,0,31
1,SAL50002,41,56,606
2,SAL50003,0,0,29
3,SAL50004,0,8,26
4,SAL50005,0,0,14


### 2.6 Process Labour Force Status (G46A & G46B)
To further understand the socio-economic drivers of housing stress, we extract labour force status data. Using the aggregated person totals from G46B, we extract counts for full-time employment, part-time employment, and unemployment to calculate future participation and unemployment rates.

In [31]:
# Create copy of G46B which contains the 'P' (Persons) aggregated totals
lfs_df = dfs['G46B'].copy()

# Define labour force categories for extraction using verified header names
# Use 'P' columns to get combined totals directly
lfs_targets = {
    'total_employed_full_time': 'P_Emp_FullT_Tot',
    'total_employed_part_time': 'P_Emp_PartT_Tot',
    'total_unemployed': 'P_Tot_Unemp_Tot',
    'total_labour_force': 'P_Tot_LF_Tot'
}

# Extract and rename columns for clarity
lfs_df = lfs_df[['SAL_CODE_2021'] + list(lfs_targets.values())]
lfs_df.rename(columns={v: k for k, v in lfs_targets.items()}, inplace=True)

# Validate output
display(lfs_df[['SAL_CODE_2021', 'total_employed_full_time', 'total_unemployed', 'total_labour_force']].head())

,SAL_CODE_2021,total_employed_full_time,total_unemployed,total_labour_force
0,SAL50001,25,0,51
1,SAL50002,335,17,640
2,SAL50003,19,0,46
3,SAL50004,17,0,28
4,SAL50005,11,0,18


### 2.7 Process Selected Characteristics (G01 & G02)
To normalize our industry and housing data into meaningful ratios, we extract total population and household counts. G01 provides the total person count, while G02 provides the average household size and median age, which serve as foundational denominators for the analysis.

In [32]:
# Create copies of G01 and G02
pop_df = dfs['G01'].copy()
avg_df = dfs['G02'].copy()

# Extract total population from G01
# Tot_P_P represents Total Persons
pop_df = pop_df[['SAL_CODE_2021', 'Tot_P_P']]
pop_df.rename(columns={'Tot_P_P': 'total_population'}, inplace=True)

# Extract average characteristics from G02
# Focus on median age and average household size
avg_cols = {
    'Median_age_persons': 'median_age',
    'Average_household_size': 'avg_household_size'
}
avg_df = avg_df[['SAL_CODE_2021'] + list(avg_cols.keys())]
avg_df.rename(columns=avg_cols, inplace=True)

# Merge characteristics into a single reference dataframe
char_df = pd.merge(pop_df, avg_df, on='SAL_CODE_2021')

# Validate output
display(char_df.head())

,SAL_CODE_2021,total_population,median_age,avg_household_size
0,SAL50001,83,39,3.0
1,SAL50002,1321,42,2.7
2,SAL50003,92,42,2.8
3,SAL50004,41,38,2.8
4,SAL50005,51,34,2.8
